# 02 Data Audit: merge, harmonize, cohort, missingness

Turns the per-component parquet files from notebook 01 into one analysis-ready adult cohort
linked to mortality. No imputation and no modeling here. Imputation happens in R (notebook 05) so the
survey design is respected.

**What this notebook decides, and the defaults it uses** (change them in the cells below if you disagree):

| Decision | Default | Why |
|---|---|---|
| Population | Adults 20+, MEC examined, eligible for mortality linkage, not pregnant | Standard NHANES mortality cohort; lab values need the exam |
| Follow-up clock | Months from MEC exam (`permth_exm`) | Labs and BP are measured at the exam, not the interview |
| Hypertension | SBP 140+ or DBP 90+ or on BP medication (JNC7) | Standard in mortality cohort literature; ACC/AHA 130/80 kept as a sensitivity flag |
| Diabetes | Told by doctor, insulin, pills, HbA1c 6.5+, or fasting glucose 126+ | ADA criteria plus self-report |
| Kidney function | eGFR from CKD-EPI 2021 (race-free) on calibrated creatinine | Current KDIGO/NKF standard |
| Weights | 1999-2002: 4-year MEC weight x 2/10; 2003-2018: 2-year MEC weight x 1/10 | CDC rule for combining 10 cycles |

**Outputs**
- `data/processed/analytic_cohort.parquet`
- `outputs/tables/02_cohort_funnel.csv`, `02_variable_coverage.csv`, `02_missingness_by_cycle.csv`, `02_range_checks.csv`

In [1]:
import pandas as pd, numpy as np, pathlib

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = pathlib.Path("/content/drive/MyDrive/NHANES Cardiometabolic Mortality Project")
except ImportError:
    ROOT = pathlib.Path.cwd().parent

assert (ROOT / "notebooks").exists(), f"Project folder not found at {ROOT}"
PARQUET_DIR, PROCESSED_DIR, TABLES_DIR = ROOT / "data" / "parquet", ROOT / "data" / "processed", ROOT / "outputs" / "tables"
for d in (PROCESSED_DIR, TABLES_DIR):
    d.mkdir(parents=True, exist_ok=True)

raw = {p.stem: pd.read_parquet(p) for p in sorted(PARQUET_DIR.glob("*.parquet"))}
pd.Series({k: f"{v.shape[0]:,} rows x {v.shape[1]} cols" for k, v in raw.items()})

Mounted at /content/drive


,0
albumin_creatinine,"81,913 rows x 16 cols"
biochemistry_profile,"69,132 rows x 51 cols"
blood_pressure_exam,"96,766 rows x 31 cols"
body_measures,"96,766 rows x 55 cols"
bp_cholesterol_questionnaire,"63,592 rows x 49 cols"
demographics,"101,316 rows x 175 cols"
diabetes_questionnaire,"96,811 rows x 81 cols"
fasting_glucose,"33,332 rows x 13 cols"
glycohemoglobin,"69,132 rows x 3 cols"
hdl_cholesterol,"82,539 rows x 9 cols"


## 1. Fix the SAS transport zero artefact

`pandas.read_sas` reads a stored zero in an XPT file as roughly `5.4e-79`. Harmless for most
columns, but it breaks any rule that tests `== 0` (a diastolic reading of 0 is a real NHANES value
that means "not measurable"). Values below `1e-70` are set to exactly 0.

In [2]:
for df in raw.values():
    num = df.select_dtypes("number").columns
    df[num] = df[num].mask(df[num].abs() < 1e-70, 0)

## 2. Harmonize variables across cycles

Each analysis variable maps to a list of candidate NHANES column names, because some names changed
between cycles (HDL is `LBDHDL` in 1999-2002, `LBXHDD` in 2003-2004, `LBDHDD` from 2005). For every
row the first non-missing candidate wins.

In [3]:
SPEC = {
    "demographics": {"sex": ["RIAGENDR"], "age": ["RIDAGEYR"], "race_eth": ["RIDRETH1"], "education": ["DMDEDUC2"],
                     "pir": ["INDFMPIR"], "pregnant": ["RIDEXPRG"], "exam_status": ["RIDSTATR"],
                     "psu": ["SDMVPSU"], "strata": ["SDMVSTRA"], "wt_mec2": ["WTMEC2YR"], "wt_mec4": ["WTMEC4YR"]},
    "body_measures": {"bmi": ["BMXBMI"], "waist": ["BMXWAIST"]},
    "blood_pressure_exam": {**{f"sbp{i}": [f"BPXSY{i}"] for i in range(1, 5)}, **{f"dbp{i}": [f"BPXDI{i}"] for i in range(1, 5)}},
    "glycohemoglobin": {"hba1c": ["LBXGH"]},
    "fasting_glucose": {"fpg": ["LBXGLU"], "wt_fast2": ["WTSAF2YR"], "wt_fast4": ["WTSAF4YR"]},
    "total_cholesterol": {"tc": ["LBXTC"]},
    "hdl_cholesterol": {"hdl": ["LBDHDD", "LBXHDD", "LBDHDL"]},
    "triglycerides_ldl": {"tg": ["LBXTR"], "ldl": ["LBDLDL"]},
    "albumin_creatinine": {"u_alb": ["URXUMA"], "u_cr": ["URXUCR"]},
    "biochemistry_profile": {"scr": ["LBXSCR", "LBDSCR"]},
    "diabetes_questionnaire": {"dm_told": ["DIQ010"], "dm_insulin": ["DIQ050"], "dm_pills": ["DIQ070", "DID070"]},
    "bp_cholesterol_questionnaire": {"htn_told": ["BPQ020"], "htn_med": ["BPQ050A"], "chol_told": ["BPQ080"], "chol_med": ["BPQ100D"]},
    "medical_conditions": {"chf": ["MCQ160B"], "chd": ["MCQ160C"], "angina": ["MCQ160D"], "mi": ["MCQ160E"], "stroke": ["MCQ160F"], "cancer": ["MCQ220"]},
    "smoking": {"smk_100": ["SMQ020"], "smk_now": ["SMQ040"]},
}

def harmonize(label, spec):
    df = raw[label]
    out = pd.DataFrame({"SEQN": df["SEQN"].astype("int64")})
    for var, cands in spec.items():
        present = [c for c in cands if c in df.columns]
        out[var] = df[present].bfill(axis=1).iloc[:, 0].to_numpy() if present else np.nan
    return out.drop_duplicates("SEQN")

merged = harmonize("demographics", SPEC["demographics"]).assign(cycle=raw["demographics"]["cycle"].to_numpy())
assert merged["SEQN"].is_unique, "SEQN repeats across cycles"
for label, spec in SPEC.items():
    if label != "demographics":
        merged = merged.merge(harmonize(label, spec), on="SEQN", how="left", validate="one_to_one")

mort = raw["mortality"].drop(columns="cycle").rename(columns={"seqn": "SEQN", "diabetes": "mcod_diabetes", "hyperten": "mcod_hypertension"}).astype({"SEQN": "int64"})
merged = merged.merge(mort, on="SEQN", how="left", validate="one_to_one")
merged.shape

(101316, 56)

### Coverage check

Share of participants with a non-missing value, by variable and cycle, for **all ages**, so
adult-only items look low. What matters here is a **0.0**: that means a harmonization gap (the
column name for that cycle isn't in `SPEC`), not a real absence. Any row printed below needs fixing
before moving on.

In [4]:
coverage = merged.drop(columns="SEQN").groupby("cycle").agg(lambda s: s.notna().mean()).T.round(3)
coverage.to_csv(TABLES_DIR / "02_variable_coverage.csv")
coverage[(coverage == 0).any(axis=1)]

cycle,1999-2000,2001-2002,2003-2004,2005-2006,2007-2008,2009-2010,2011-2012,2013-2014,2015-2016,2017-2018
wt_mec4,1.000,1.000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
wt_fast4,0.328,0.332,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Expected zeros, not bugs: `wt_mec4` and `wt_fast4` exist only for 1999-2000 and 2001-2002 (4-year weights).

Anything else at 0.0 means that cycle stores the variable under a column name missing from `SPEC`.
Look up the name in the CDC variable list for that cycle and add it to the candidate list. Two such
renames are already handled: serum creatinine is `LBDSCR` in 2001-2002, and diabetes pills are
`DID070` in 2005-2008.

## 3. Derive analysis variables

NHANES questionnaire codes: 1 = yes, 2 = no, 7 = refused, 9 = don't know. Refused and don't know
become missing. Several items are only asked after a "yes" (e.g. diabetes pills are asked only of people
told they have diabetes), so a missing value there counts as "no" when building the composite
conditions, not as unknown.

In [5]:
d = merged.copy()
yn = lambda s: s.map({1: 1.0, 2: 0.0})
for c in ["dm_insulin", "dm_pills", "htn_told", "htn_med", "chol_told", "chol_med", "chf", "chd", "angina", "mi", "stroke", "cancer"]:
    d[c] = yn(d[c])
d["dm_borderline"] = d["dm_told"].eq(3).astype(float).where(d["dm_told"].isin([1, 2, 3]))
d["dm_told"] = d["dm_told"].map({1: 1.0, 2: 0.0, 3: 0.0})
d["education"] = d.education.where(d.education.between(1, 5))

d["sbp"] = d[[f"sbp{i}" for i in range(1, 5)]].mean(axis=1)
d["dbp"] = d[[f"dbp{i}" for i in range(1, 5)]].replace(0, np.nan).mean(axis=1)

def flag(conds, known):
    hit = pd.concat(conds, axis=1).any(axis=1)
    return pd.Series(np.where(hit, 1.0, np.where(known, 0.0, np.nan)), index=d.index)

d["diabetes"] = flag([d.dm_told.eq(1), d.dm_insulin.eq(1), d.dm_pills.eq(1), d.hba1c.ge(6.5), d.fpg.ge(126)], d.hba1c.notna() & d.dm_told.notna())
d["prediabetes"] = flag([d.hba1c.between(5.7, 6.4), d.fpg.between(100, 125), d.dm_borderline.eq(1)], d.hba1c.notna()).where(d.diabetes.ne(1), 0.0)
d["glycemic_status"] = np.select([d.diabetes.eq(1), d.prediabetes.eq(1), d.diabetes.eq(0) & d.prediabetes.eq(0)], ["diabetes", "prediabetes", "normal"], None)

d["hypertension"] = flag([d.sbp.ge(140), d.dbp.ge(90), d.htn_med.eq(1)], d.sbp.notna())
d["hypertension_acc"] = flag([d.sbp.ge(130), d.dbp.ge(80), d.htn_med.eq(1)], d.sbp.notna())

d["bmi_cat"] = pd.cut(d.bmi, [0, 18.5, 25, 30, np.inf], right=False, labels=["underweight", "normal", "overweight", "obese"])
d["non_hdl"] = d.tc - d.hdl
d["acr"] = d.u_alb / d.u_cr * 100
d["albuminuria"] = d.acr.ge(30).astype(float).where(d.acr.notna())

### eGFR

Serum creatinine in 1999-2000 and 2005-2006 was measured on assays that drifted from the
standardized reference, so CDC and the published literature apply these corrections before any eGFR:

- 1999-2000: standard = 1.013 x creatinine + 0.147
- 2005-2006: standard = 0.978 x creatinine - 0.016

eGFR then uses the race-free CKD-EPI 2021 equation (Inker et al., NEJM 2021). CKD = eGFR below 60
or albuminuria (ACR 30+ mg/g).

In [6]:
d["scr_std"] = np.select([d.cycle.eq("1999-2000"), d.cycle.eq("2005-2006")], [1.013 * d.scr + 0.147, 0.978 * d.scr - 0.016], d.scr)
female = d.sex.eq(2)
r = d.scr_std / np.where(female, 0.7, 0.9)
d["egfr"] = 142 * np.minimum(r, 1) ** np.where(female, -0.241, -0.302) * np.maximum(r, 1) ** -1.200 * 0.9938 ** d.age * np.where(female, 1.012, 1)
d["ckd"] = flag([d.egfr.lt(60), d.albuminuria.eq(1)], d.egfr.notna() | d.acr.notna())

In [7]:
d["cvd_history"] = flag([d[c].eq(1) for c in ["chf", "chd", "angina", "mi", "stroke"]], d[["chf", "chd", "angina", "mi", "stroke"]].notna().any(axis=1))
d["smoking"] = np.select([d.smk_100.eq(2), d.smk_100.eq(1) & d.smk_now.isin([1, 2]), d.smk_100.eq(1) & d.smk_now.eq(3)], ["never", "current", "former"], None)
d["sex_lbl"] = d.sex.map({1: "male", 2: "female"})
d["race_eth_lbl"] = d.race_eth.map({1: "Mexican American", 2: "Other Hispanic", 3: "Non-Hispanic White", 4: "Non-Hispanic Black", 5: "Other/Multiracial"})
d["education_lbl"] = d.education.map({1: "<HS", 2: "<HS", 3: "HS/GED", 4: ">HS", 5: ">HS"})
d["age_group"] = pd.cut(d.age, [20, 40, 60, 80, np.inf], right=False, labels=["20-39", "40-59", "60-79", "80+"])

### Outcomes and weights

Cause of death uses the NCHS leading-cause recode (`ucod_leading`): 1 heart disease and 5
cerebrovascular are grouped as **CVD**, 2 as **cancer**, everything else as **other**. A death
with no cause recorded is kept as **unknown** rather than folded into "other".

In [8]:
d["time_months"], d["death"] = d.permth_exm, d.mortstat
d["cause"] = np.select([d.mortstat.eq(0), d.ucod_leading.isin([1, 5]), d.ucod_leading.eq(2), d.ucod_leading.notna(), d.mortstat.eq(1)], ["alive", "cvd", "cancer", "other", "unknown"], None)
early = d.cycle.isin(["1999-2000", "2001-2002"])
d["wt_mec"] = np.where(early, d.wt_mec4 * 2 / 10, d.wt_mec2 / 10)
d["wt_fast"] = np.where(early, d.wt_fast4 * 2 / 10, d.wt_fast2 / 10)

## 4. Cohort definition

Every exclusion is counted. The printed table is the cohort flow for the report and dashboard.

In [9]:
RULES = [
    ("Age 20+", lambda x: x.age >= 20),
    ("MEC examined", lambda x: x.exam_status == 2),
    ("Eligible for mortality linkage", lambda x: x.eligstat == 1),
    ("Not pregnant", lambda x: x.pregnant.ne(1)),
    ("Follow-up time and status available", lambda x: x.time_months.notna() & x.death.notna()),
]
cohort, funnel = d, [("All NHANES participants 1999-2018", len(d))]
for step, rule in RULES:
    cohort = cohort[rule(cohort)]
    funnel.append((step, len(cohort)))
funnel = pd.DataFrame(funnel, columns=["step", "n_remaining"]).assign(n_excluded=lambda f: (-f.n_remaining.diff()).fillna(0).astype(int))
funnel.to_csv(TABLES_DIR / "02_cohort_funnel.csv", index=False)
funnel

,step,n_remaining,n_excluded
0,All NHANES participants 1999-2018,101316,0
1,Age 20+,55081,46235
2,MEC examined,52398,2683
3,Eligible for mortality linkage,52287,111
4,Not pregnant,50819,1468
5,Follow-up time and status available,50819,0


In [10]:
display(pd.Series({
    "participants": len(cohort),
    "deaths": int(cohort.death.sum()),
    "median follow-up (years)": round(cohort.time_months.median() / 12, 1),
    "person-years": round(cohort.time_months.sum() / 12),
    "weighted population (millions)": round(cohort.wt_mec.sum() / 1e6, 1),
}, name="value").to_frame())
cohort.cause.value_counts().rename("n").to_frame()

,value
participants,50819.0
deaths,8279.0
median follow-up (years),9.0
person-years,479971.0
weighted population (millions),213.7


,n
cause,
alive,42540
other,3848
cvd,2593
cancer,1837
unknown,1


### Are the linkage-ineligible adults different?

Adults excluded for mortality linkage lacked enough identifying information to match against the
National Death Index. If they differ a lot from the retained group, that's a limitation to state.

In [11]:
adults = d[(d.age >= 20) & (d.exam_status == 2)]
adults.groupby(adults.eligstat.eq(1).map({True: "linked", False: "not linkable"})).agg(
    n=("SEQN", "size"), mean_age=("age", "mean"), pct_female=("sex", lambda s: s.eq(2).mean() * 100),
    pct_diabetes=("diabetes", lambda s: s.mean() * 100), mean_bmi=("bmi", "mean")).round(1)

,n,mean_age,pct_female,pct_diabetes,mean_bmi
eligstat,,,,,
linked,52287,49.7,51.9,17.2,28.9
not linkable,111,46.7,57.7,14.1,28.0


## 5. Missingness in the analytic cohort

Percent missing per variable by cycle. `fpg`, `tg`, `ldl` are fasting-subsample only (about half
of adults by design), so high missingness there is expected and handled with the fasting weight,
not imputation.

In [12]:
KEY = ["age", "sex", "race_eth", "education", "pir", "bmi", "waist", "sbp", "dbp", "hba1c", "fpg", "tc", "hdl", "tg", "ldl",
       "acr", "egfr", "diabetes", "hypertension", "cvd_history", "cancer", "smoking", "psu", "strata", "wt_mec"]
missing = cohort.groupby("cycle")[KEY].agg(lambda s: s.isna().mean() * 100).T.round(1)
missing["overall"] = cohort[KEY].isna().mean().mul(100).round(1)
missing.to_csv(TABLES_DIR / "02_missingness_by_cycle.csv")
missing.sort_values("overall", ascending=False)

cycle,1999-2000,2001-2002,2003-2004,2005-2006,2007-2008,2009-2010,2011-2012,2013-2014,2015-2016,2017-2018,overall
ldl,60.3,58.2,56.4,56.6,55.3,55.2,54.9,54.8,59.1,54.9,56.5
tg,54.6,53.5,55.1,55.4,54.1,54.4,54.1,54.1,58.7,54.5,54.8
fpg,54.1,52.6,55.0,54.9,54.0,54.0,53.5,53.7,55.1,53.7,54.0
pir,14.3,7.1,5.8,4.7,9.2,9.6,8.5,7.7,10.3,12.9,9.0
egfr,7.3,6.5,5.9,6.6,6.8,6.2,8.0,4.6,5.8,6.5,6.4
hdl,7.4,6.7,5.5,6.2,6.5,5.9,7.5,4.3,5.6,6.1,6.1
tc,7.4,6.7,5.4,6.2,6.5,5.9,7.5,4.3,5.6,6.1,6.1
waist,3.8,5.7,5.9,5.8,5.8,5.8,6.4,5.6,6.4,6.0,5.8
dbp,4.5,5.9,8.0,5.9,5.4,4.8,4.7,3.7,3.1,5.2,5.1
hba1c,5.4,5.0,4.5,5.7,6.1,5.0,5.5,3.4,4.8,4.5,5.0


## 6. Range checks

Values outside physiologically plausible limits are counted, not dropped. They are CDC exam and lab
measurements taken under quality control, and only a handful are flagged, so they stay in the data.
Their influence on the models is limited in R by modelling continuous measures with splines rather
than straight lines.

In [13]:
LIMITS = {"bmi": (12, 80), "waist": (40, 200), "sbp": (60, 270), "dbp": (20, 150), "hba1c": (3, 20), "fpg": (40, 600),
          "tc": (50, 700), "hdl": (5, 200), "tg": (10, 5000), "egfr": (1, 200), "acr": (0, 20000), "age": (20, 85)}
checks = pd.DataFrame([{"variable": v, "low": lo, "high": hi, "n_below": int(cohort[v].lt(lo).sum()), "n_above": int(cohort[v].gt(hi).sum()),
                        "min": cohort[v].min(), "max": cohort[v].max()} for v, (lo, hi) in LIMITS.items()])
checks.to_csv(TABLES_DIR / "02_range_checks.csv", index=False)
checks

,variable,low,high,n_below,n_above,min,max
0,bmi,12,80,0,7,12.040000,130.210000
1,waist,40,200,0,0,55.500000,178.200000
2,sbp,60,270,0,0,64.666667,270.000000
3,dbp,20,150,24,0,3.333333,138.666667
4,hba1c,3,20,1,0,2.000000,18.800000
5,fpg,40,600,6,0,21.000000,587.300000
6,tc,50,700,0,3,59.000000,813.000000
7,hdl,5,200,0,1,6.000000,226.000000
8,tg,10,5000,0,0,10.000000,4233.000000
9,egfr,1,200,0,1,2.015401,241.962700


Note on age: NHANES top-codes age at 85 for 1999-2006 and at 80 from 2007. The `age` column
is therefore capped, and the `80+` age group is the only safe way to use the oldest participants.

## 7. Save the analytic cohort

In [14]:
KEEP = ["SEQN", "cycle", "psu", "strata", "wt_mec", "wt_fast",
        "age", "age_group", "sex", "sex_lbl", "race_eth", "race_eth_lbl", "education", "education_lbl", "pir", "smoking",
        "bmi", "bmi_cat", "waist", "sbp", "dbp", "hba1c", "fpg", "tc", "hdl", "non_hdl", "tg", "ldl", "acr", "scr_std", "egfr",
        "dm_told", "dm_insulin", "dm_pills", "htn_told", "htn_med", "chol_told", "chol_med",
        "diabetes", "prediabetes", "glycemic_status", "hypertension", "hypertension_acc", "albuminuria", "ckd",
        "chf", "chd", "angina", "mi", "stroke", "cvd_history", "cancer",
        "time_months", "death", "cause", "ucod_leading", "mcod_diabetes", "mcod_hypertension"]
out = cohort[KEEP]
out.to_parquet(PROCESSED_DIR / "analytic_cohort.parquet", index=False)
out.shape

(50819, 58)

## Next: notebook 03, DuckDB SQL

Business questions against `analytic_cohort.parquet`: crude death rates by cardiometabolic
profile, prevalence by cycle, cause-of-death mix by age group. Crude numbers only; the
survey-weighted estimates come from R.